# AAI6640 2026 Spring Assignment 3

# Section One: Hyperparameter Tuning

In [60]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import pandas as pd 
from itertools import product

BATCH_SIZE = 64

# 1. Load MNIST Dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
full_train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# split the full train into train & val
train_size = int(0.9 * len(full_train_data))
val_size = len(full_train_data) - train_size
train_data, val_data = random_split(full_train_data, [train_size, val_size])

# data loaders
full_train_loader = DataLoader(full_train_data, batch_size=BATCH_SIZE, shuffle=True)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=1000)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)


print("Imports and data loading / preprocessing complete")
print("=" * 50)


print(f"Full Train Dataset Size: {len(full_train_data)}")
print(f"Train Dataset Size: {len(train_data)}")
print(f"Validation Dataset Size: {len(val_data)}")
print(f"Test Dataset Size: {len(test_data)}")


Imports and data loading / preprocessing complete
Full Train Dataset Size: 60000
Train Dataset Size: 54000
Validation Dataset Size: 6000
Test Dataset Size: 10000


## <div class="alert alert-info">[GRADED  TASK 1.1]</div>
Please build an MLP model for the MNIST dataset and tune the following hyperparameters:
- "hidden_layers": [[128], [256, 128], [512, 256, 128]]
- "dropout": [0.2, 0.3, 0.5]
- "lr": [0.01, 0.001, 0.0005]

For each combination of the above hyperparameters, train the model on __train_data__ for 20 epochs.

Report the best model and the corresponding validation accuracy on __val_data__.

Then, train the best model on __full_train_data__ for 20 epochs, and report the test accuracy on __test_data__.

In [62]:
# Your answer here
'''
MLP architecture:
- Input layer: 784 neurons (28x28 pixels flattened)
- Hidden layer 1: (784, 256) neurons + ReLU activation -> dropout(0.3)
- Hidden layer 2: (256, 128) neurons + ReLU activation -> dropout(0.3)
- Output layer: (128, 10) neurons (for 10 classes) + no activation (logits)

Implemented a class MLP that inherits from nn.Module. I chose this architecture because its common for image classification tasks like MINST.
- super is used to call the __init__ method of the parent class nn.Module to ensure proper initialization of the model
- the forward method helps with the foward pass of the model, helps with he input flow through the layers and activations, returns the output logits for classification

- The Final layer needs to be linear because it will use CrossEntropy Loss  which combines LogSoftMax and NLLLoss in a single class.
    Expects raw logits to later convert them to probabilities and compute the loss. 


Notes: 
- realize i created a global variable for input size and output, but it will only update the 1st hidden layer and no longer will be 784. 
    So for the second hidden layer to work, i will need to create a new local variable in forward pass to keep track. I also don't want to update the global variable

References:
Python Built-ins:

zip() — https://docs.python.org/3/library/functions.html#zip
itertools.product — https://docs.python.org/3/library/itertools.html#itertools.product
dict.get() (the .get(key, default) pattern) — https://docs.python.org/3/library/stddict.html#dict.get
'''
 
# creating a dictionary to store the model parameters and call them in the MLP class
model_params = {
    'input_size': 28*28,
    'output_size': 10, # number of classes in MNIST 
    'hidden_layers_grid': [[128], [256, 128], [512, 256, 128]],
    'dropout_rate_grid': [0.2, 0.3, 0.5],
    'lr_grid': [0.01, 0.001, 0.0005],
    'num_epochs': 20
}

class MLP(nn.Module):
    def __init__(self, hidden_layers, dropout):
        super(MLP, self).__init__() # used to call the parent class constructor 
        
        # flatten the model input 
        self.flatten = nn.Flatten() # used to flatten the input image from 28x28 to 784 
        
         # ------ Build Hidden Layers ------
        self.hidden_layers = nn.ModuleList() # used to store hidden layers in a list. nn built in module instead of a regular list 
        self.dropout = nn.ModuleList() # used to store dropout layers in a list. nn built in module instead of a regular list
        
        # initialize input features to the input size of the model (784 for MNIST). Need local variable so 784 does not change and only applied to the 1st hidden layer
        input_features = model_params['input_size'] 
        
        # input layer and dropout into hidden layers 
        for hidden_size in hidden_layers:
            self.hidden_layers.append(nn.Linear(input_features, hidden_size)) # this inputs the flattened image into the 1st hidden layer 
            
            # do the same for the dropout layerse 
            self.dropout.append(nn.Dropout(dropout))
            
            # update and output 
            input_features = hidden_size # updating the input features for the next hidden layer until output is reached
            
        # ------ Output Layer ------
        self.output_layer = nn.Linear(input_features, model_params['output_size']) # the output layer that takes the last hidden layer output and produces the final output for classification
        
    # Forward pass, used to define how the input data flows through the network of hiddne layers and dropout layers for regularization and then to the output layer
    def forward(self, x):
        '''
        - Forward passing through the network 
        - Will apply ReLU activation at each hidden layer to introduce non-linearity and dropout for regularization. Helps prevent overfitting and reduces size of each hidden layer
        - x (Tensor): is the input data, or batch of images, that will be passed through
            - batch size x 1 x 28 x 28 (for MNIST)
        - Returns: the output after passing through. the raw logits (scores) for each class
            - batch size x 10 (for MNIST, 10 classes)

        --------------
       
        '''
        
        
        
        # flatten the input image 
        x = self.flatten(x) # flatten the image from 28x28 to 784 
        
        # -------- ReLU Activation and dropout for hidden layers --------
        # instead of writing out each hidden layer and dropout seperately like past assignment code, will use zip() to loop through and pair them together
        for h_layer, drop_out in zip(self.hidden_layers, self.dropout):
            x = torch.relu(h_layer(x))
            x = drop_out(x)
        # Output layer 
        x = self.output_layer(x) 
        return x # returning x, the raw logits (scores) for each class 

print("MLP class deinfed and compiled successfully")
print("=" * 50)
print()


MLP class deinfed and compiled successfully



In [63]:
# Testing MLP class and its hidden layers and dropout for the model builds
for test_config in model_params['hidden_layers_grid']:
    for dropout_rate in model_params['dropout_rate_grid']:
        #print(f'Built MLP with hidden layers: {test_config} and dropout rate: {dropout_rate}')
        test_model = MLP(test_config, dropout_rate)
        test_input = torch.randn(2 ,1,28,28) # creating a random input tensor batch of 2 images, 1 channel, and 28x28 pixels
        test_output = test_model(test_input) # passing the test input through the model to get output
print("=" * 80)
print("Test have successfully ran for MLP hidden layers and dropout configurations ")
print("=" * 80)
print(test_model)
print("=" * 80)
print(f'Output shape: {test_output.shape}')


Test have successfully ran for MLP hidden layers and dropout configurations 
MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): Linear(in_features=512, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=128, bias=True)
  )
  (dropout): ModuleList(
    (0-2): 3 x Dropout(p=0.5, inplace=False)
  )
  (output_layer): Linear(in_features=128, out_features=10, bias=True)
)
Output shape: torch.Size([2, 10])


In [66]:
''' This section will be used for training and evaluation. For conistency, I will reuse the same training and evalaution techniques
    from Assignment 2. But in this case i will seperate the functions for read ability and modularity. Also since we are implmenting grid search
    we need to be able to call the training and evaluation functions when needed for each configuration, instead of trying to call from a single function.

Errors: 
- was creating a brand new model inside the training function by calling model = MLP() which was a mistake because it would not update the 
model weights and would just create a new model each time. I removed that line and now the training function takes in the model as an argument also this will help complete the tests for training and evaluation functions.
'''

# ----------- TRAIN THE MODEL -----------
def model_train_one_epoch(model, train_loader, optimizer, criterion, epoch):    
    model.train()                       # set model to training mode
    epoch_loss = 0.0              # track cumulative loss for the epoch

 
    # get the loss function for classification
    criterion = nn.CrossEntropyLoss()
        
    # iterate over the training data in batches using the train_loader
    for input, labels in train_loader:
            
        # optimize the model weights 
        optimizer.zero_grad()          # zero the gradients before backpropagation, clear old gradients
            
        # output the model
        output = model(input)
            
        # compute cross-entropy loss between model output and true labels for classification MINST 0-9 
        loss = criterion(output, labels)
            
        # optimize the model weights 
        loss.backward()                # compute gradients based on the loss
           
        # update the models weights based on gradient compute and optimization
        optimizer.step()
            
        # update loss counter for the epoch
        epoch_loss += loss.item()
        
    # Calculate the average loss of each epoch and update the training losses list 
    avg_loss = epoch_loss / len(train_loader)
    print('From training function:')
    print(f'Epoch {epoch}, Loss: {avg_loss:.4f}')
    return avg_loss # return the average loss for the epoch
    


# ----------- EVALUATE THE MODEL -----------
    
def evaluate_model(model, test_loader):
                
       
    # evalaute the model on the test set after each epoch and store the accuracy in the accuracies list 
    model.eval()
    correct = 0
    total = 0 
    with torch.no_grad():
        for target_input, target_labels in test_loader:
            target_output = model(target_input)
            _, predicted = target_output.max(dim=1)
            total += target_labels.size(0)
            correct += predicted.eq(target_labels.view_as(predicted)).sum().item()
            
        # compute and track the accuracy
        accuracy = 100.0 * correct /total

    
    # return accuracy for the epoch
    return accuracy

print("Training and evaluation complete!")


Training and evaluation complete!


In [67]:
'''Testing Train and Evaluate functions'''


# ============================TEST MODEL ON TRAIN AND EVALUATE======================================================
# test the model_train_one_epoch and evaluate_model functions 
tester_model = MLP(model_params['hidden_layers_grid'][1], model_params['dropout_rate_grid'][1]) # using the 2nd config for testing
tester_optimizer = optim.Adam(tester_model.parameters(), lr=0.001)
tester_criterion = nn.CrossEntropyLoss()

#training for 1 epoch
tester_loss = model_train_one_epoch(tester_model, train_loader, tester_optimizer, tester_criterion, epoch=1)

print("=" * 50)

# Evaluate the model on validation data and track test accruacies and losses 
test_accuracies = evaluate_model(tester_model, test_loader)
print(test_accuracies)
print(model_params['hidden_layers_grid'][1], model_params['dropout_rate_grid'][1])
print(tester_loss)
print("-" * 50)
print("Successfully trained and evaluated the model tracked for each epoch!")   
print("-" * 50)
print(f'Final Loss: {tester_loss:.4f}, Final accuracy: {test_accuracies:.2f}%')     
            

From training function:
Epoch 1, Loss: 0.3179
95.86
[256, 128] 0.3
0.3178742668851857
--------------------------------------------------
Successfully trained and evaluated the model tracked for each epoch!
--------------------------------------------------
Final Loss: 0.3179, Final accuracy: 95.86%


In [68]:
'''For each combination of the above hyperparameters, train the model on __train_data__ for 20 epochs.

Report the best model and the corresponding validation accuracy on __val_data__.

Then, train the best model on __full_train_data__ for 20 epochs, and report the test accuracy on __test_data__.


Goal: Since we are attempting to search every possible combination of hyperparameters, we will loop through each combination of hidden layers, dropout,lr. 
 - Grid Search best fits this problem 

Since we need combinations of all types since there are 3 options of each hyperparameter, it will take 27 iterations total
    (3 hidden layers x 3 dropout rates x 3 learning rates = 27 (1 choice from each list)) to search through all combinations
    
    - instead of creating a nested loop for each hyperparameter like in the early testing of the MLP class.
    can use itertools.product to create a single loop to iterate through all combinations
    and enumerate to keep track of each iteration and epoch.
'''

combination_results = [] # list to store the different combinations of hyperparameters with their corresponding validation accuracies and losses for each epoch

# calculate the total number of combinations for tracking progress
'''instead of hardcoding 27, calculated based on hyperparam length from model_params dictionary. 
# This ensures we truly are calculating the right total count of combinations, instead of assuming its 27.'''
total_combinations = len(model_params['hidden_layers_grid']) * len(model_params['dropout_rate_grid']) * len(model_params['lr_grid'])
print(f'Total Combinations: {total_combinations}')
print("=" * 50)

# iterate through all combinations of hyperparameters using itertools.product and using enumerate to track(count) the combinations 
for count_combos, (hidden_layers, dropout, lr) in enumerate(
    product(
        model_params['hidden_layers_grid'],
        model_params['dropout_rate_grid'],
        model_params['lr_grid']
    ), 1 # start the count at 1 instead of 0. If we start a 0 we will have to add +1 and also finish at 26 total instead of 27
):
    print(f'Combination {count_combos}/{total_combinations} | Hidden Layers: {hidden_layers}, Dropout: {dropout}, Learning Rate: {lr}' )
    
    
    # --------- Create a new model for each combination of hyperparameters ---------
    '''This will help keep the model from carrying over the previous models training and evaluation'''
    
    new_model = MLP(hidden_layers, dropout) 
    new_optimizer = optim.Adam(new_model.parameters(), lr=0.001)
    new_criterion = nn.CrossEntropyLoss()
    
    # ------- Train the Model ----------
    # count epochs 
    epoch_plus_one = model_params['num_epochs'] + 1 # adding +1 moves to the next position epoch
    # loop through the number of epochs, (1,) is the starting point for epoch count and to finish at 20 instead of 19, using +1 to iterate to the next epoch
    for epoch in range(1, epoch_plus_one):
        model_train_one_epoch(new_model, train_loader, new_optimizer, new_criterion, epoch)
        
    # ------- Evaluate the Model ---------
    # Evaluating the validation data: tracking accuracy for each combination 
    validation_acc = evaluate_model(new_model, val_loader)
    print(f'Validation Accuracy: {validation_acc:.4f}')
    
    # --------- Store the results for each combination ---------
    combination_results.append({
        'Hidden Layers': str(hidden_layers),
        'Dropout Rate': dropout,
        'Learning Rate': lr,
        'Validation Accuracy': f'{validation_acc:.4f}'
    })

Total Combinations: 27
Combination 1/27 | Hidden Layers: [128], Dropout: 0.2, Learning Rate: 0.01
From training function:
Epoch 1, Loss: 0.3062
From training function:
Epoch 2, Loss: 0.1500
From training function:
Epoch 3, Loss: 0.1154
From training function:
Epoch 4, Loss: 0.0968
From training function:
Epoch 5, Loss: 0.0838
From training function:
Epoch 6, Loss: 0.0735
From training function:
Epoch 7, Loss: 0.0682
From training function:
Epoch 8, Loss: 0.0624
From training function:
Epoch 9, Loss: 0.0590
From training function:
Epoch 10, Loss: 0.0550
From training function:
Epoch 11, Loss: 0.0522
From training function:
Epoch 12, Loss: 0.0494
From training function:
Epoch 13, Loss: 0.0449
From training function:
Epoch 14, Loss: 0.0433
From training function:
Epoch 15, Loss: 0.0384
From training function:
Epoch 16, Loss: 0.0399
From training function:
Epoch 17, Loss: 0.0384
From training function:
Epoch 18, Loss: 0.0353
From training function:
Epoch 19, Loss: 0.0363
From training func

KeyboardInterrupt: 

In [ ]:
'''
Report the best model and the corresponding validation accuracy on __val_data__.

'''

# convert comparison data dictionary into a dataframe for better visualization and to sort
comparison_df = pd.DataFrame(combination_results)

# sort the dataframe by Accuracy in descending order
comparison_df = comparison_df.sort_values(by="Validation Accuracy", ascending=False).reset_index(drop=True) # reset index after sorting and drop the old index

print('\n' + "=" * 50)
print("Model Performance Comparison: Hypterparameter Tuning Results of 27 Combinations")
print("=" * 50)

# display comparison table with no index
print(comparison_df.to_string(index=False))
print("-" * 80)


# ----- Best model configuration -----
print("\nBest Model Configuration Combination:\n")
best_performing_combo = comparison_df.iloc[0]
print(f"Hidden Layers: {best_performing_combo['Hidden Layers']}")
print(f"Dropout: {best_performing_combo['Dropout Rate']}")
print(f"Learning Rate: {best_performing_combo['Learning Rate']}")
print(f"Validation Accuracy: {best_performing_combo['Validation Accuracy']}")


    


Model Performance Comparison: Hypterparameter Tuning Results of 27 Combinations
  Hidden Layers  Dropout Rate  Learning Rate Validation Accuracy
[512, 256, 128]           0.2         0.0005             98.4000
     [256, 128]           0.2         0.0100             98.3333
[512, 256, 128]           0.2         0.0010             98.2833
[512, 256, 128]           0.3         0.0100             98.2333
[512, 256, 128]           0.3         0.0005             98.2167
     [256, 128]           0.3         0.0005             98.1667
[512, 256, 128]           0.5         0.0100             98.1500
[512, 256, 128]           0.2         0.0100             98.1500
     [256, 128]           0.3         0.0010             98.1333
     [256, 128]           0.5         0.0100             98.0500
     [256, 128]           0.2         0.0010             98.0333
[512, 256, 128]           0.3         0.0010             98.0167
[512, 256, 128]           0.5         0.0005             97.9667
         

In [ ]:
'''This section will train the best model configuration on the full training data and evaluate on the test data
- Best Model Configurations:
    - the best config will be automaticalled pulled from the grid search results 
    
Full Train Dataset Size: 60000
Train Dataset Size: 54000
Validation Dataset Size: 6000
Test Dataset Size: 10000

The model will have the full amount of the data to train on and learn before the final evaluation on the test data.
'''

# =================== Retraining with the Best Model Configurations on Full Train Data ======================

# ------ Best hyperparameters config from the grid search ------
best_hidden_layers = best_performing_combo['Hidden Layers']
best_dropout = best_performing_combo['Dropout Rate']
best_learning_rate = best_performing_combo['Learning Rate']


# ------ Build a fresh model with best hyperparameters ------
winning_model = MLP(eval(best_hidden_layers), best_dropout) # added eval() to convert the string back to a list for hidden layers. It was giving me an error.
winning_optimizer = optim.Adam(winning_model.parameters(), lr=best_learning_rate)
winning_criterion = nn.CrossEntropyLoss()

# ------- Train the Model ----------
# count epochs 
epoch_plus_one = model_params['num_epochs'] + 1 # adding +1 moves to the next position epoch
# loop through the number of epochs, (1,) is the starting point for epoch count and to finish at 20 instead of 19, using +1 to iterate to the next epoch
for epoch in range(1, epoch_plus_one):
   training_loss = model_train_one_epoch(winning_model, full_train_loader, winning_optimizer, winning_criterion, epoch)
        
# ------- Evaluate the Model ---------
# Evaluating the validation data: tracking accuracy for each combination 
validation_acc = evaluate_model(winning_model, test_loader) # use the test data to evaluate the final model performance
print(f'Validation Accuracy: {validation_acc:.4f}')

From training function:
Epoch 1, Loss: 0.3040
From training function:
Epoch 2, Loss: 0.1244
From training function:
Epoch 3, Loss: 0.0919
From training function:
Epoch 4, Loss: 0.0735
From training function:
Epoch 5, Loss: 0.0629
From training function:
Epoch 6, Loss: 0.0547
From training function:
Epoch 7, Loss: 0.0467
From training function:
Epoch 8, Loss: 0.0451
From training function:
Epoch 9, Loss: 0.0375
From training function:
Epoch 10, Loss: 0.0353
From training function:
Epoch 11, Loss: 0.0334
From training function:
Epoch 12, Loss: 0.0326
From training function:
Epoch 13, Loss: 0.0292
From training function:
Epoch 14, Loss: 0.0275
From training function:
Epoch 15, Loss: 0.0246
From training function:
Epoch 16, Loss: 0.0242
From training function:
Epoch 17, Loss: 0.0233
From training function:
Epoch 18, Loss: 0.0211
From training function:
Epoch 19, Loss: 0.0223
From training function:
Epoch 20, Loss: 0.0218
Validation Accuracy: 98.2700


In [58]:
print(best_hidden_layers)

[512, 256, 128]


# Section Two - Comparing Normalizations

## <div class="alert alert-info">[GRADED  TASK 2.1]</div>
Using the best MLP model found above, let's compare the following normalization methods:
- BatchNorm
- LayerNorm
- GroupNorm
- No Normalization

The goal is to identify which normalization method achieves the highest validation accuracy.

In [51]:
'''
References:
- PyTorch Documentation for nn.Module: https://pytorch.org/docs/stable/generated/torch
PyTorch Normalization Layers:

nn.BatchNorm1d — https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html
nn.LayerNorm — https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html
nn.GroupNorm — https://pytorch.org/docs/stable/generated/torch.nn.GroupNorm.html
nn.Identity — https://pytorch.org/docs/stable/generated/torch.nn.Identity.html

'''


model_params = {
    'input_size': 28*28,
    'output_size': 10, # number of classes in MNIST 
    'hidden_layers_grid': [[128], [256, 128], [512, 256, 128]],
    'dropout_rate_grid': [0.2, 0.3, 0.5],
    'lr_grid': [0.01, 0.001, 0.0005],
    'num_epochs': 20
}

class NormMLP(nn.Module):
    def __init__(self, hidden_layers, dropout, norm_type=None):
        super(NormMLP, self).__init__() # used to call the parent class constructor 
        
        # flatten the model input 
        self.flatten = nn.Flatten() # used to flatten the input image from 28x28 to 784 
        
         # ------ Build Hidden Layers ------
        self.hidden_layers = nn.ModuleList() # used to store hidden layers in a list. nn built in module instead of a regular list 
        self.dropout = nn.ModuleList() # used to store dropout layers in a list. 
        self.norm = nn.ModuleList() # used to store normalization layers in a list.
        
        # initialize input features to the input size of the model (784 for MNIST). Need local variable so 784 does not change and only applied to the 1st hidden layer
        input_features = model_params['input_size'] 
        
        # input layer and dropout into hidden layers 
        for hidden_size in hidden_layers:
            self.hidden_layers.append(nn.Linear(input_features, hidden_size)) # this inputs the flattened image into the 1st hidden layer 
            
            # adding normalization layer after the linear layer and before ReLU activation
            # this is needed to help the model stabilize and converge faster during training 
            
            norm_options = {
                'batch': nn.BatchNorm1d(hidden_size), # Batch normalization normalizes the acvtivations of the previous layer for each batch. Good for generalization
                'layer': nn.LayerNorm(hidden_size), # Layer normalization normalizes across the features for each data point
                'group': nn.GroupNorm(8, hidden_size) # Group Normalization divides the channels into groups then normalizes with each group
            }
            
            # implement normalization 
            self.norm.append(norm_options.get(norm_type, nn.Identity())) # nn.Identity() used a default normalization layer if nothing is passed for norm_type or invalid
            
            # do the same for the dropout layerse 
            self.dropout.append(nn.Dropout(dropout))
            
            # update and output 
            input_features = hidden_size # updating the input features for the next hidden layer until output is reached
            
        # ------ Output Layer ------
        self.output_layer = nn.Linear(input_features, model_params['output_size']) # the output layer that takes the last hidden layer output and produces the final output for classification
        
    # Forward pass, used to define how the input data flows through the network of hiddne layers and dropout layers for regularization and then to the output layer
    def forward(self, x):
        '''
        - Forward passing through the network 
        - The Forward pass will now go from:
            - Original MLP forward pass:
                - Linear -> ReLU -> Dropout 
            - Norm MLP forward pass 
                - Linear -> Norm -> ReLU -> Dropout
       
        '''
        
        # flatten the input image 
        x = self.flatten(x) # flatten the image from 28x28 to 784 
        
        # -------- ReLU Activation and dropout for hidden layers --------
        # instead of writing out each hidden layer and dropout seperately like past assignment code, will use zip() to loop through and pair them together
        for h_layer, norm, drop_out in zip(self.hidden_layers, self.norm, self.dropout):
            x = h_layer(x) # linear layer
            x = norm(x) # normalization comes after linear layer and before activation
            x = torch.relu(x) 
            x = drop_out(x)
        # Output layer 
        x = self.output_layer(x) 
        return x # returning x, the raw logits (scores) for each class 

print("NormMLP class defined and compiled successfully")
print("=" * 50)
print()


NormMLP class defined and compiled successfully



In [53]:
# Testing NormMLP class and its hidden layers and dropout for the model builds
for norm in ['batch', 'layer', 'group'] :
    for test_config in model_params['hidden_layers_grid']:
        for drop_out in model_params['dropout_rate_grid']:
            test_model = NormMLP(test_config, drop_out, norm)
            test_input = torch.randn(2 ,1,28,28) # creating a random input tensor batch of 2 images, 1 channel, and 28x28 pixels
            test_output = test_model(test_input) # passing the test input through the model to get output
print("=" * 80)
print("Test have successfully ran for NormMLP hidden layers and dropout configurations ")
print("=" * 80)
print(test_model)
print("=" * 80)
print(f'Output shape: {test_output.shape}')




Test have successfully ran for NormMLP hidden layers and dropout configurations 
NormMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): Linear(in_features=512, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=128, bias=True)
  )
  (dropout): ModuleList(
    (0-2): 3 x Dropout(p=0.5, inplace=False)
  )
  (norm): ModuleList(
    (0): GroupNorm(8, 512, eps=1e-05, affine=True)
    (1): GroupNorm(8, 256, eps=1e-05, affine=True)
    (2): GroupNorm(8, 128, eps=1e-05, affine=True)
  )
  (output_layer): Linear(in_features=128, out_features=10, bias=True)
)
Output shape: torch.Size([2, 10])


In [59]:
'''
This section will be used to implement grid search on the NormMLP Model and then compare the results against the MLP model

'''

# store results 
norm_model_results = []

# ------ Best hyperparameters config from the grid search ------
best_hidden_layers = best_performing_combo['Hidden Layers']
best_dropout = best_performing_combo['Dropout Rate']
best_learning_rate = best_performing_combo['Learning Rate']

for norm in ['batch', 'layer', 'group', 'none']:
    # ------ Build a fresh model with best hyperparameters ------
    norm_mlp_model = NormMLP(eval(best_hidden_layers), best_dropout, norm) # added eval() to convert the string back to a list for hidden layers. It was giving me an error.
    norm_mlp_optimizer = optim.Adam(norm_mlp_model.parameters(), lr=best_learning_rate)
    norm_mlp_criterion = nn.CrossEntropyLoss()

    # ------- Train the Model ----------
    # count epochs 
    epoch_plus_one = model_params['num_epochs'] + 1 # adding +1 moves to the next position epoch
    # loop through the number of epochs, (1,) is the starting point for epoch count and to finish at 20 instead of 19, using +1 to iterate to the next epoch
    for epoch in range(1, epoch_plus_one):
        training_loss = model_train_one_epoch(norm_mlp_model, full_train_loader, norm_mlp_optimizer, norm_mlp_criterion, epoch)
            
    # ------- Evaluate the Model ---------
    # Evaluating the validation data: tracking accuracy for each combination 
    validation_acc = evaluate_model(norm_mlp_model, test_loader) # use the test data to evaluate the final model performance
    print(f'Validation Accuracy: {validation_acc:.4f}')
    
    # store the results for each normalization type
    norm_model_results.append({
        'Hidden Layers': best_hidden_layers,
        'Dropout Rate': best_dropout,
        'Learning Rate': best_learning_rate,
        'Normalization Type': norm,
        'Validation Accuracy': f'{validation_acc:.4f}',
        'Final Loss': f'{training_loss:.4f}'
    })



From training function:
Epoch 1, Loss: 0.2940
From training function:
Epoch 2, Loss: 0.1242
From training function:
Epoch 3, Loss: 0.0982
From training function:
Epoch 4, Loss: 0.0817
From training function:
Epoch 5, Loss: 0.0690
From training function:
Epoch 6, Loss: 0.0613
From training function:
Epoch 7, Loss: 0.0541
From training function:
Epoch 8, Loss: 0.0496
From training function:
Epoch 9, Loss: 0.0463
From training function:
Epoch 10, Loss: 0.0402
From training function:
Epoch 11, Loss: 0.0373
From training function:
Epoch 12, Loss: 0.0314
From training function:
Epoch 13, Loss: 0.0335
From training function:
Epoch 14, Loss: 0.0295
From training function:
Epoch 15, Loss: 0.0287
From training function:
Epoch 16, Loss: 0.0261
From training function:
Epoch 17, Loss: 0.0273
From training function:
Epoch 18, Loss: 0.0233
From training function:
Epoch 19, Loss: 0.0236
From training function:
Epoch 20, Loss: 0.0219
Validation Accuracy: 98.5600
From training function:
Epoch 1, Loss: 0

In [57]:
print(best_hidden_layers)

[512, 256, 128]


In [ ]:
# convert comparison data dictionary into a dataframe for better visualization and to sort
norm_comparison_df = pd.DataFrame(norm_model_results)

# sort the dataframe by Accuracy in descending order
norm_comparison_df = norm_comparison_df.sort_values(by="Validation Accuracy", ascending=False).reset_index(drop=True) # reset index after sorting and drop the old index

print('\n' + "=" * 50)
print("NormMLP Model Performnace Comparison with different Normalization Types in a DataFrame")
print("=" * 50)

# display comparison table with no index
print(norm_comparison_df.to_string(index=False))
print("-" * 80)



NormMLP Model Performnace Comparison with different Normalization Types in a DataFrame
  Hidden Layers  Dropout Rate  Learning Rate Normalization Type Validation Accuracy Final Loss
[512, 256, 128]           0.5         0.0005              batch             98.5400     0.0223
[512, 256, 128]           0.5         0.0005              group             98.4100     0.0222
[512, 256, 128]           0.5         0.0005              layer             98.3900     0.0227
[512, 256, 128]           0.5         0.0005               none             98.2400     0.0193
--------------------------------------------------------------------------------


In [71]:
# Compare the best NormMLP model against the best MLP model

#empty list to store the best model results for comparison
best_model_comparison = []

# best NormMLP model
best_norm_model = norm_comparison_df.iloc[0]
print("\nBest NormMLP Model Configuration Combination:\n")
print(f"Hidden Layers: {best_norm_model['Hidden Layers']}")
print(f"Dropout: {best_norm_model['Dropout Rate']}")
print(f"Learning Rate: {best_norm_model['Learning Rate']}")
print(f"Normalization Type: {best_norm_model['Normalization Type']}")
print(f"Validation Accuracy: {best_norm_model['Validation Accuracy']}")

# Best MLP model from grid search 
best_mlp_model = comparison_df.iloc[0]
print("\nBest MLP Model Configuration Combination:\n")
print(f"Hidden Layers: {best_mlp_model['Hidden Layers']}")
print(f"Dropout: {best_mlp_model['Dropout Rate']}")
print(f"Learning Rate: {best_mlp_model['Learning Rate']}")
print(f"Validation Accuracy: {best_mlp_model['Validation Accuracy']}")

# append the best models to the comparison list for final display
for best_model in [best_norm_model, best_mlp_model]:
    best_model_comparison.append({
        'Model': 'NormMLP' if best_model is best_norm_model else 'MLP',
        'Hidden Layers': best_model['Hidden Layers'],
        'Dropout': best_model['Dropout Rate'],
        'Learning Rate': best_model['Learning Rate'],
        'Normalization Type': best_model['Normalization Type'] if best_model is best_norm_model else 'none',
        'Validation Accuracy': best_model['Validation Accuracy']
    })

# turn data into dataframe 
final_df = pd.DataFrame(best_model_comparison)
# sort by Accuracy Score 
final_df = final_df.sort_values(by='Validation Accuracy', ascending=False)

print("=" * 50)
print("Top Model Config Comparison Results")
print("=" * 50)

# display comparison table with no index
print(final_df.to_string(index=False))
print("-" * 80)


Best NormMLP Model Configuration Combination:

Hidden Layers: [512, 256, 128]
Dropout: 0.5
Learning Rate: 0.0005
Normalization Type: batch
Validation Accuracy: 98.5400

Best MLP Model Configuration Combination:

Hidden Layers: [512, 256, 128]
Dropout: 0.2
Learning Rate: 0.0005
Validation Accuracy: 98.4000
Top Model Config Comparison Results
  Model   Hidden Layers  Dropout  Learning Rate Normalization Type Validation Accuracy
NormMLP [512, 256, 128]      0.5         0.0005              batch             98.5400
    MLP [512, 256, 128]      0.2         0.0005               none             98.4000
--------------------------------------------------------------------------------
